# theprotocol.it

In [ ]:
# httpx get html
import httpx
from bs4 import BeautifulSoup
url = "https://theprotocol.it/filtry/python;t/big-data-science,ai-ml;sp/junior,assistant,trainee;p/zdalna;rw"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "pl-PL,pl;q=0.9,en-US;q=0.8,en;q=0.7"
}
protocol = httpx.get(url, headers=headers)
soup = BeautifulSoup(protocol.text, 'html.parser')
backdrop = soup.find("div", {"data-test": "backdrop"})
if backdrop:
    backdrop.parent.decompose()
scripts = soup.find_all("script")
for script in scripts:
    script.decompose()
print(soup)

In [ ]:
# httpx get api whole site
import httpx

def get_protocol_csrf_token():
    """Get XSRF-TOKEN cookie from /csrf-token endpoint."""
    url = "https://apus-api.theprotocol.it/csrf-token"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    with httpx.Client() as client:
        client.get(url, headers=headers)
        # print(client.cookies)
        return client.cookies
    
def fetch_protocol_offers(page_number=80, page_size=50):
    """Fetch job offers from theprotocol.it API for a given page."""
    url = f"https://apus-api.theprotocol.it/offers/_search?pageNumber={page_number}&orderby.field=Relevance&pageSize={page_size}"
    cookies = get_protocol_csrf_token()
    xsrf_token = cookies.get("XSRF-TOKEN")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Content-Type": "application/json",
        "x-xsrf-token": xsrf_token
    }
    with httpx.Client(cookies=cookies) as client:
        response = client.post(url, headers=headers)

        if not response.text:
            return None
        return response.json()['offers']

offers = fetch_protocol_offers()
print(len(offers))
# import json
# offers_json = json.dumps(offers, indent=4, ensure_ascii=False)
# print(offers_json)

## Get API response from URL

In [ ]:
# backup - 2 first links working
import httpx
from urllib.parse import urlparse, unquote, parse_qs

def get_protocol_csrf_token():
    url = "https://apus-api.theprotocol.it/csrf-token"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    with httpx.Client() as client:
        client.get(url, headers=headers)
        return client.cookies

def _parse_position_levels(part: str) -> list:
    """Parse position levels from 'sp/' part. Returns list of level IDs."""
    poziomy = {
        "junior": 17,
        "assistant": 3,
        "trainee": 1,
        "mid": 2,
        "senior": 4,
        "lead": 5,
    }
    return [poziomy[p] for p in part[3:].split(",") if p in poziomy]

def _parse_work_modes_and_cities(part: str, work_modes: dict) -> tuple:
    """Parse work modes and cities from 'p/' part. Returns tuple (work_modes, cities)."""
    values = [v.lower() for v in part[2:].split(",") if v]
    work_mode_codes = [work_modes[v] for v in values if v in work_modes]
    cities = [v.capitalize() for v in values if v not in work_modes]
    return work_mode_codes, cities

def _parse_wp_work_modes(part: str, work_modes: dict) -> list:
    """Parse work modes from 'wp/' part. Returns list of work mode codes."""
    values = [v.lower() for v in part[3:].split(",") if v]
    return [work_modes[v] for v in values if v in work_modes]

def _parse_technologies(part: str) -> list:
    """Parse technologies from part. Returns list of technologies."""
    return [t.replace("%23", "#") for t in part.split(",")]

def _parse_specializations(part: str) -> list:
    """Parse specializations from 't/' part. Returns list of codes."""
    return part[2:].split(",")

def _parse_excluded_technologies(query: dict) -> list:
    """Parse excluded technologies from query dict. Returns list of technologies."""
    return [t.upper() if t.lower() == "r" else t for t in query.get("et", [])]

def parse_protocol_url(url: str) -> dict:
    """Parse theprotocol.it search URL and build API payload. Input: URL string. Output: dict for API POST."""
    from urllib.parse import urlparse, unquote, parse_qs
    parsed = urlparse(url)
    path = unquote(parsed.path)
    query = parse_qs(parsed.query)
    payload = {
        "typesOfContractIds": [],
        "positionLevelIds": [],
        "cities": [],
        "workModeCodes": [],
        "onlyWithProjectDescription": False,
        "technologies": [],
        "excludedTechnologies": [],
        "regionsOfWorld": [],
        "keywords": [],
        "specializationsCodes": [],
        "isSupportingUkraine": False,
        "fromExternalLocations": False
    }
    work_modes = {
        "zdalna": "home-office",
        "hybrydowa": "hybrid",
        "stacjonarna": "full-office"
    }
    if "/filtry/" in path:
        filter_str = path.split("/filtry/")[1]
        parts = filter_str.split(";")
        for part in parts:
            if part.startswith("t/"):
                payload["specializationsCodes"] = _parse_specializations(part)
            elif part.startswith("sp/"):
                payload["positionLevelIds"] = _parse_position_levels(part)
            elif part.startswith("p/"):
                work_mode_codes, cities = _parse_work_modes_and_cities(part, work_modes)
                payload["workModeCodes"].extend(work_mode_codes)
                payload["cities"].extend(cities)
            elif part.startswith("wp/"):
                payload["workModeCodes"].extend(_parse_wp_work_modes(part, work_modes))
            elif part:
                payload["technologies"].extend(_parse_technologies(part))
    if "et" in query:
        payload["excludedTechnologies"] = _parse_excluded_technologies(query)
        payload["fromExternalLocations"] = True
    return payload


def fetch_protocol_offers_from_url(url: str, page_number=1, page_size=50):
    cookies = get_protocol_csrf_token()
    xsrf_token = cookies.get("XSRF-TOKEN")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Content-Type": "application/json",
        "x-xsrf-token": xsrf_token,
        "x-application-name": "theprotocol-offers",
        "x-application-version": "4.4.1169",
        "Origin": "https://theprotocol.it",
        "Referer": url
    }
    payload = parse_protocol_url(url)
    api_url = f"https://apus-api.theprotocol.it/offers/_search?pageNumber={page_number}&orderby.field=Relevance&pageSize={page_size}"
    with httpx.Client(cookies=cookies) as client:
        response = client.post(api_url, headers=headers, json=payload)
        if not response.text:
            return []
        return response.json().get('offers', [])

# Przykład użycia:
url = "https://theprotocol.it/filtry/python;t/big-data-science,ai-ml,data-analytics-bi;sp/junior,assistant,trainee;p/zdalna;rw"
offers1 = fetch_protocol_offers_from_url(url)
print(len(offers1))

url2 = "https://theprotocol.it/filtry/c++,c%23;t/ai-ml,backend,qa-testing,security,frontend;sp/junior,assistant,trainee;p/warszawa;wp/stacjonarna,hybrydowa;rw?et=r"
offers2 = fetch_protocol_offers_from_url(url2)
print(len(offers2))

url3 = "https://theprotocol.it/filtry/c++,c,angular;t/ai-ml,backend,qa-testing,security,frontend,helpdesk,it-admin,data-analytics-bi,big-data-science,ux-ui,devops;sp/junior,assistant,trainee,mid,manager;p/kontrakt-b2b,umowa-zlecenie,umowa-na-zastepstwo;c/1000;s/warszawa,wroclaw;wp/stacjonarna,hybrydowa;rw?et=r&et=c%2523"
offers3 = fetch_protocol_offers_from_url(url3)
print(len(offers3))


4
9
0


In [4]:
import json
print(json.dumps(offers1, indent=4, ensure_ascii=False))

[
    {
        "id": "dbe30000-d4f2-5aeb-2919-08dddb11dab4",
        "groupId": "bf7342f0-ed78-f011-b481-6045bd9c0d1b",
        "title": "Data Science Trainee",
        "employer": "EPAM Systems (Poland) sp. z o.o.",
        "employerId": "20097714",
        "logoUrl": "https://logos.gpcdn.pl/loga-firm/20097714/9b030000-5dac-0015-6883-08db89155d81_280x280.png",
        "offerUrlName": "data-science-trainee-krakow-fabryczna-1a,oferta,dbe30000-d4f2-5aeb-2919-08dddb11dab4",
        "aboutProject": [
            "Striving to gain market-oriented knowledge and skills to jumpstart your career in IT? Apply for this program and shape your professional path with EPAM experts.",
            "If you are eager to thrive in the data-driven world and master one of the most trending professions, then this expert-led program is what you need.",
            "By participating, you will have the opportunity to:",
            "•\tGain a strong foundation in software engineering",
            "•\tDive dee

In [ ]:
url1 = "https://theprotocol.it/filtry/python;t/big-data-science,ai-ml,data-analytics-bi;sp/junior,assistant,trainee;p/zdalna;rw"
expected_len_1 = 4
print(len(offers1) == expected_len_1)

url2 = "https://theprotocol.it/filtry/c++,c%23;t/ai-ml,backend,qa-testing,security,frontend;sp/junior,assistant,trainee;p/warszawa;wp/stacjonarna,hybrydowa;rw?et=r"
expected_len_2 = 9
print(len(offers2) == expected_len_2)

url3 = "https://theprotocol.it/filtry/c++,c,angular;t/ai-ml,backend,qa-testing,security,frontend,helpdesk,it-admin,data-analytics-bi,big-data-science,ux-ui,devops;sp/junior,assistant,trainee,mid,manager;p/kontrakt-b2b,umowa-zlecenie,umowa-na-zastepstwo;c/1000;s/warszawa,wroclaw;wp/stacjonarna,hybrydowa;rw?et=r&et=c%2523"
expected_len_3 = 8
print(len(offers3) == expected_len_3)

False
True


# WORKING
3 links tested - need more\
**need refactor**

In [ ]:
import httpx
from urllib.parse import urlparse, unquote, parse_qs

def get_protocol_csrf_token():
    url = "https://apus-api.theprotocol.it/csrf-token"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
    }
    with httpx.Client() as client:
        client.get(url, headers=headers)
        return client.cookies

def _parse_position_levels(part: str) -> list:
    """Parse position levels from 'sp/' part. Returns list of level IDs."""
    poziomy = {
        "trainee": 1,
        "assistant": 3,
        "mid": 4,
        "lead": 5,
        "head": 6,
        "junior": 17,
        "senior": 18,
        "expert": 19,
        "manager": 20,
    }
    levels = part[3:].split(",")
    return [poziomy[p] for p in levels if p in poziomy]

def _parse_work_modes_and_cities(part: str, work_modes: dict) -> tuple:
    """Parse work modes and cities from 'p/' part. Returns tuple (work_modes, cities)."""
    values = [v.lower() for v in part[2:].split(",") if v]
    work_mode_codes = [work_modes[v] for v in values if v in work_modes]
    cities = [v.capitalize() for v in values if v not in work_modes]
    return work_mode_codes, cities

def _parse_wp_work_modes(part: str, work_modes: dict) -> list:
    """Parse work modes from 'wp/' part. Returns list of work mode codes."""
    values = [v.lower() for v in part[3:].split(",") if v]
    return [work_modes[v] for v in values if v in work_modes]

def _parse_technologies(part: str) -> list:
    """Parse technologies from part. Returns list of technologies with correct casing."""
    tech_map = {
        "python": "Python",
        "c++": "C++",
        "c": "C",
        "angular": "Angular",
        "javascript": "Javascript",
        "sql": "SQL",
        "delphi": "Delphi",
        "wordpress": "Wordpress",
        "c#": "C#",
        "php": "PHP",
        # Dodaj inne technologie według potrzeb
    }
    return [tech_map.get(t.replace("%23", "#").lower(), t.replace("%23", "#").capitalize()) for t in part.split(",")]

def _parse_specializations(part: str) -> list:
    """Parse specializations from 't/' part. Returns list of codes mapped to API codes."""
    mapping = {
        "qa-testing": "testing",
        "data-analytics-bi": "data-analytics-and-bi",
        "big-data-science": "big-data-science",
        "ai-ml": "ai-ml",
        "frontend": "frontend",
        "backend": "backend",
        "security": "security",
        "helpdesk": "helpdesk",
        "it-admin": "it-admin",
        "ux-ui": "ux-ui",
        "devops": "devops"
    }
    return [mapping.get(code, code) for code in part[2:].split(",")]

def _parse_excluded_technologies(query: dict) -> list:
    """Parse excluded technologies from query dict. Returns list of technologies."""
    result = []
    for t in query.get("et", []):
        if t.lower() == "r":
            result.append("R")
        else:
            decoded = unquote(unquote(t)).replace("%23", "#")
            if decoded.lower() == "c#":
                result.append("C#")
            else:
                result.append(decoded)
    return result

def _parse_types_of_contract(part: str) -> list:
    """Parse contract types from 'p/' part. Returns list of contract type IDs."""
    contract_types = {
        "kontrakt-b2b": 3,
        "umowa-zlecenie": 2,
        "umowa-na-zastepstwo": 4,
        "umowa-o-prace": 1,
        "umowa-o-dzielo": 5,
        "praktyki-staz": 6,
    }
    return [contract_types[t] for t in part[2:].split(",") if t in contract_types]

def _parse_salary_from(part: str) -> str:
    """Parse salary from 'c/' part. Returns salaryFrom as string."""
    return part[2:] if part[2:].isdigit() else ""

def _parse_cities(part: str) -> list:
    """Parse cities from 's/' part. Returns list of cities with correct Polish letters."""
    city_map = {
        "warszawa": "Warszawa",
        "krakow": "Kraków",
        "wroclaw": "Wrocław",
        "szczecin": "Szczecin",
        "poznan": "Poznań",
        "lublin": "Lublin",
        "katowice": "Katowice",
        "bydgoszcz": "Bydgoszcz",
        "lodz": "Łódź",
        "bialystok": "Białystok",
        "torun": "Toruń",
        "gdansk": "Gdańsk"
    }
    return [city_map.get(city.lower(), city.capitalize()) for city in part[2:].split(",") if city]

def parse_protocol_url(url: str) -> dict:
    """Parse theprotocol.it search URL and build API payload. Input: URL string. Output: dict for API POST."""
    parsed = urlparse(url)
    path = unquote(parsed.path)
    query = parse_qs(parsed.query)
    payload = {
        "typesOfContractIds": [],
        "positionLevelIds": [],
        "cities": [],
        "workModeCodes": [],
        "onlyWithProjectDescription": False,
        "technologies": [],
        "excludedTechnologies": [],
        "regionsOfWorld": [],
        "keywords": [],
        "specializationsCodes": [],
        "isSupportingUkraine": False,
        "fromExternalLocations": False
    }
    work_modes = {
        "zdalna": "home-office",
        "hybrydowa": "hybrid",
        "stacjonarna": "full-office"
    }
    if "/filtry/" in path:
        filter_str = path.split("/filtry/")[1]
        parts = filter_str.split(";")
        for part in parts:
            if part.startswith("t/"):
                payload["specializationsCodes"] = _parse_specializations(part)
            elif part.startswith("sp/"):
                payload["positionLevelIds"] = _parse_position_levels(part)
            elif part.startswith("p/") and any(x in part for x in ["kontrakt-b2b", "umowa-zlecenie", "umowa-na-zastepstwo", "umowa-o-prace", "umowa-o-dzielo", "praktyki-staz"]):
                payload["typesOfContractIds"] = _parse_types_of_contract(part)
            elif part.startswith("p/"):
                work_mode_codes, cities = _parse_work_modes_and_cities(part, work_modes)
                payload["workModeCodes"].extend(work_mode_codes)
                payload["cities"].extend(cities)
            elif part.startswith("wp/"):
                payload["workModeCodes"].extend(_parse_wp_work_modes(part, work_modes))
            elif part.startswith("c/"):
                salary = _parse_salary_from(part)
                if salary:
                    payload["salaryFrom"] = salary
            elif part.startswith("s/"):
                payload["cities"].extend(_parse_cities(part))
            elif part:
                payload["technologies"].extend(_parse_technologies(part))
    if "et" in query:
        excluded = _parse_excluded_technologies(query)
        payload["excludedTechnologies"] = excluded
        payload["fromExternalLocations"] = True
    return payload
def fetch_protocol_offers_from_url(url: str, page_number=1, page_size=50):
    cookies = get_protocol_csrf_token()
    xsrf_token = cookies.get("XSRF-TOKEN")
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36",
        "Content-Type": "application/json",
        "x-xsrf-token": xsrf_token,
        "x-application-name": "theprotocol-offers",
        "x-application-version": "4.4.1169",
        "Origin": "https://theprotocol.it",
        "Referer": url
    }
    
    payload = parse_protocol_url(url)
    print(payload)
    api_url = f"https://apus-api.theprotocol.it/offers/_search?pageNumber={page_number}&orderby.field=Relevance&pageSize={page_size}"
    with httpx.Client(cookies=cookies) as client:
        response = client.post(api_url, headers=headers, json=payload)
        if not response.text:
            return []
        return response.json().get('offers', [])

# Przykład użycia:
url = "https://theprotocol.it/filtry/python;t/big-data-science,ai-ml,data-analytics-bi;sp/junior,assistant,trainee;p/zdalna;rw"
offers1 = fetch_protocol_offers_from_url(url)
# print(len(offers1))

url2 = "https://theprotocol.it/filtry/c++,c%23;t/ai-ml,backend,qa-testing,security,frontend;sp/junior,assistant,trainee;p/warszawa;wp/stacjonarna,hybrydowa;rw?et=r"
offers2 = fetch_protocol_offers_from_url(url2)
# print(len(offers2))

url3 = "https://theprotocol.it/filtry/c++,c,angular;t/ai-ml,qa-testing,security,frontend,helpdesk,it-admin,data-analytics-bi,big-data-science,ux-ui,devops,backend;sp/junior,assistant,trainee,mid,manager;p/kontrakt-b2b,umowa-zlecenie,umowa-na-zastepstwo;c/1000;s/warszawa,wroclaw;wp/stacjonarna,hybrydowa;rw?et=r&et=c%2523"
offers3 = fetch_protocol_offers_from_url(url3)
# print(len(offers3))

{'typesOfContractIds': [], 'positionLevelIds': [17, 3, 1], 'cities': [], 'workModeCodes': ['home-office'], 'onlyWithProjectDescription': False, 'technologies': ['Python'], 'excludedTechnologies': [], 'regionsOfWorld': [], 'keywords': [], 'specializationsCodes': ['big-data-science', 'ai-ml', 'data-analytics-and-bi'], 'isSupportingUkraine': False, 'fromExternalLocations': False}
4
{'typesOfContractIds': [], 'positionLevelIds': [17, 3, 1], 'cities': ['Warszawa'], 'workModeCodes': ['full-office', 'hybrid'], 'onlyWithProjectDescription': False, 'technologies': ['C++', 'C#'], 'excludedTechnologies': ['R'], 'regionsOfWorld': [], 'keywords': [], 'specializationsCodes': ['ai-ml', 'backend', 'testing', 'security', 'frontend'], 'isSupportingUkraine': False, 'fromExternalLocations': True}
9
{'typesOfContractIds': [3, 2, 4], 'positionLevelIds': [17, 3, 1, 4, 20], 'cities': ['Warszawa', 'Wrocław'], 'workModeCodes': ['full-office', 'hybrid'], 'onlyWithProjectDescription': False, 'technologies': ['C++

In [23]:
import json
print(json.dumps(offers3, indent=2, ensure_ascii=False))

[
  {
    "id": "dbe30000-d4f2-5aeb-ccdc-08dde3c088bb",
    "groupId": "0137eea3-9c81-f011-b485-6045bd9c00f8",
    "title": "Frontend Developer Angular / React",
    "employer": "Flotman Sp. z o.o.",
    "employerId": "20092349",
    "logoUrl": "https://logos.gpcdn.pl/loga-firm/20092349/03000000-bb2f-3863-9be2-08d8d26cc6d7_280x280.png",
    "offerUrlName": "frontend-developer-angular---react-warszawa-wincentego-rzymowskiego-53,oferta,dbe30000-d4f2-5aeb-ccdc-08dde3c088bb",
    "aboutProject": [
      "W ramach swoich obowiązków będziesz odpowiedzialny za rozwój głównej aplikacji webowej, a także za rozbudowę aplikacji mobilnych tworzonych w technologii Ionic z wykorzystaniem Angulara.",
      "Szukamy osoby samodzielnej, która potrafi przejąć inicjatywę w zakresie tworzenia warstwy frontendowej – zaczynając od analizy dostępnej architektury backendowej, przez projektowanie interfejsu, aż po implementację i optymalizację."
    ],
    "workplace": [
      {
        "location": "Kraków, Gr